<a href="https://colab.research.google.com/github/Karis-ilham/karis-ilham/blob/main/2318093KarisIlhamKlasifikasiCitraTeksturmetodeRandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**1. Connect Drive**

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#**2. Import Library**

In [11]:
import os
import cv2
import numpy as np
import pandas as pd

from skimage.feature import graycomatrix, graycoprops
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

#**3. Cek Dataset**



In [12]:
dataset_path = "/content/drive/MyDrive/Notebooks/Dataset_UTS"

for root, dirs, files in os.walk(dataset_path):
    print("Folder:", root)
    break

#**4. Load Dataset dari Drive**

In [17]:
import os
import cv2

data = []
labels = []

dataset_path = "/content/drive/MyDrive/Colab Notebooks/Dataset_UTS"

max_data = 100
count = 0

for file in os.listdir(dataset_path):

    if count >= max_data:
        break

    if file.lower().endswith(('.png', '.jpg', '.jpeg')):

        img_path = os.path.join(dataset_path, file)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is not None:
            img = cv2.resize(img, (128,128))

            data.append(img)

            # label dari nama file
            label = file.split('_')[0]
            labels.append(label)

            count += 1

print("Total data:", len(data))
print("Label unik:", set(labels))

Total data: 100
Label unik: {'normal', 'scratch', 'crack', 'hole', 'rust'}


#**5. Ektraksi Fitur GLCM**

In [18]:
from skimage.feature import graycomatrix, graycoprops
import numpy as np

features = []

for img in data:
    glcm = graycomatrix(
        img,
        distances=[1],
        angles=[0],
        levels=256,
        symmetric=True,
        normed=True
    )

    contrast = graycoprops(glcm, 'contrast')[0][0]
    correlation = graycoprops(glcm, 'correlation')[0][0]
    energy = graycoprops(glcm, 'energy')[0][0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0][0]

    features.append([contrast, correlation, energy, homogeneity])

features = np.array(features)

print("Jumlah fitur:", features.shape)

Jumlah fitur: (100, 4)


#**6. Buat Dataset**

In [19]:
import pandas as pd

df = pd.DataFrame(features, columns=[
    'contrast','correlation','energy','homogeneity'
])

df['label'] = labels

df.head()

,contrast,correlation,energy,homogeneity,label
0,6.311700,0.962839,0.072416,0.497163,crack
1,15.872170,0.957906,0.078342,0.631279,crack
2,7.958477,0.959680,0.127312,0.785802,crack
3,5.302288,0.912760,0.113450,0.566606,crack
4,9.625308,0.940544,0.094998,0.620255,crack


#**7. Metode Random Forest**

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df[['contrast','correlation','energy','homogeneity']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

#**8. Evaluasi**

In [21]:
from sklearn.metrics import accuracy_score, confusion_matrix

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Akurasi:", acc)
print("Confusion Matrix:\n", cm)

Akurasi: 0.45
Confusion Matrix:
 [[2 2 0 0 1]
 [0 4 0 2 0]
 [0 0 1 0 0]
 [1 5 0 1 0]
 [0 0 0 0 1]]


#**9. Simpan ke CSV**

In [22]:
csv_path = "/content/drive/MyDrive/Colab Notebooks/Dataset_UTS/fitur_glcm.csv"

df.to_csv(csv_path, index=False)

print("File CSV berhasil disimpan di:")
print(csv_path)

File CSV berhasil disimpan di:
/content/drive/MyDrive/Colab Notebooks/Dataset_UTS/fitur_glcm.csv
